# SwitchSim — Qubit-scaling results

This notebook performs a reproducible scaling experiment for the existing SwitchSim pipeline:

- **10 QPUs**, each with **10 internal/computational qubits** (100 total computational-qubit capacity).
- The QPUs use the existing fully connected `build_network()` model, i.e. the logical all-to-all connectivity provided by the central switching layer.
- GHZ benchmark circuits are swept from **12 to 100 qubits inclusive, in steps of 1**.
- For every circuit size: circuit preparation → pytket-dqc distribution → bridge export → Qoala scheduling → AFA-QSP → HAMFA-QSP.
- Results are checkpointed to CSV after each completed circuit size.
- Four publication-style figures are produced and exported as **PDF, SVG, and high-resolution PNG**.

The plotted EPR-pair count uses the existing `result_dqc["ebits"]` metric. The official pytket-dqc `ebits_builtin` value is retained in the data table as a cross-check.
> Compatibility: this notebook raises pytket QASM `maxwidth` for circuits above 32 qubits and provides local fallbacks for the original project's missing custom analysis hooks. The plotted EPR count uses pytket-dqc `ebit_cost`.


## 1. Setup

In [1]:
import contextlib
import io
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.config import BRIDGE_FILE
from src.kernel_bridge import start_bridge
from src.scheduler import schedule_pytket_dqc_with_qoala
from src.afa_qsp import simulate_afa_qsp
from src.hamfa_qsp import simulate_hamfa_qsp
from src.plotting import plot_all_results
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURE_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
CHECKPOINT_CSV = RESULTS_DIR / 'qubit_sweep_results.csv'
CONFIG_JSON = RESULTS_DIR / 'qubit_sweep_config.json'
PYTKET_METADATA_FILE = '/tmp/switchsim_qubit_sweep_metadata.json'
PYTKET = start_bridge()
print(f'Project root: {PROJECT_ROOT}')
print(f'Results CSV: {CHECKPOINT_CSV}')
print(f'Figures:     {FIGURE_DIR}')


Starting secondary kernel: pytket_dqc


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


✓ pytket_dqc kernel ready

 SwitchSim dual-kernel system READY
Normal cells → Qoala / NetSquid
%%pytket     → persistent pytket_dqc
%pytket_restart → restart pytket kernel
Project root: /Users/seyednavidelyasi/Library/CloudStorage/OneDrive-Chalmers/Desktop/SwitchSim_project
Results CSV: /Users/seyednavidelyasi/Library/CloudStorage/OneDrive-Chalmers/Desktop/SwitchSim_project/results/qubit_sweep_results.csv
Figures:     /Users/seyednavidelyasi/Library/CloudStorage/OneDrive-Chalmers/Desktop/SwitchSim_project/results/figures


## 2. Experiment configuration

In [2]:
DISTRIBUTION_METHOD = 'PartitioningAnnealing'
N_QPUS = 12
QUBITS_PER_QPU = 10
QUBIT_COUNTS = list(range(11, 120, 5))
BENCHMARK = 'ghz'
DISTRIBUTION_SEED = 1
t = 490
MID_CIRCUIT_MEASUREMENT_TIME_NS = 11000 + t
EPR_ATTEMPT_TIME_NS = 198981
SINGLE_QUBIT_TIME_NS = 5500
TWO_QUBIT_TIME_NS = 66000
STARTING_PROCESS_TIME_NS = EPR_ATTEMPT_TIME_NS + MID_CIRCUIT_MEASUREMENT_TIME_NS + TWO_QUBIT_TIME_NS
ENDING_PROCESS_TIME_NS = SINGLE_QUBIT_TIME_NS + TWO_QUBIT_TIME_NS + t
SCHEDULING_STRATEGY = 'QOALA'
P_SUCC = 0.044426421303161545
TIME_PER_TRIAL_NS = 198981
PROTOCOL_SEED = 42
DELTA_T_C_NS = 2800000000.0
F_I = 0.9796744718797619
F_T = 0.9306907483
CUTOFF_FRACTION = 0.05
ALPHA = CUTOFF_FRACTION
T_CUT_NS = -DELTA_T_C_NS * np.log(CUTOFF_FRACTION)
DEADLINE_EQUAL_IS_BLOCKED = True
BACKGROUND_START_NS = 0.0
CONTINUE_ON_ERROR = True
experiment_config = {'network': {'n_qpus': N_QPUS, 'qubits_per_qpu': QUBITS_PER_QPU, 'total_internal_qubits': N_QPUS * QUBITS_PER_QPU}, 'sweep': {'benchmark': BENCHMARK, 'qubit_start': QUBIT_COUNTS[0], 'qubit_stop': QUBIT_COUNTS[-1], 'qubit_step': 1, 'distribution_seed': DISTRIBUTION_SEED, 'distribution_method': DISTRIBUTION_METHOD}, 'qoala': {'single_qubit_time_ns': SINGLE_QUBIT_TIME_NS, 'two_qubit_time_ns': TWO_QUBIT_TIME_NS, 'starting_process_time_ns': STARTING_PROCESS_TIME_NS, 'ending_process_time_ns': ENDING_PROCESS_TIME_NS, 'strategy': SCHEDULING_STRATEGY}, 'qsp': {'p_succ': P_SUCC, 'time_per_trial_ns': TIME_PER_TRIAL_NS, 'protocol_seed_base': PROTOCOL_SEED, 't_cut_ns': T_CUT_NS, 'delta_t_c_ns': DELTA_T_C_NS, 'F_I': F_I, 'F_T': F_T, 'alpha': ALPHA, 'deadline_equal_is_blocked': DEADLINE_EQUAL_IS_BLOCKED, 'background_start_ns': BACKGROUND_START_NS}}
CONFIG_JSON.write_text(json.dumps(experiment_config, indent=2))
experiment_config


{'network': {'n_qpus': 12, 'qubits_per_qpu': 10, 'total_internal_qubits': 120},
 'sweep': {'benchmark': 'ghz',
  'qubit_start': 11,
  'qubit_stop': 116,
  'qubit_step': 1,
  'distribution_seed': 1,
  'distribution_method': 'PartitioningAnnealing'},
 'qoala': {'single_qubit_time_ns': 5500,
  'two_qubit_time_ns': 66000,
  'starting_process_time_ns': 276471,
  'ending_process_time_ns': 71990,
  'strategy': 'QOALA'},
 'qsp': {'p_succ': 0.044426421303161545,
  'time_per_trial_ns': 198981,
  'protocol_seed_base': 42,
  't_cut_ns': 8388050365.951175,
  'delta_t_c_ns': 2800000000.0,
  'F_I': 0.9796744718797619,
  'F_T': 0.9306907483,
  'alpha': 0.05,
  'deadline_equal_is_blocked': True,
  'background_start_ns': 0.0}}

## 3. Sweep helpers

The pytket-dqc work is sent to the persistent secondary kernel. For a scaling study, circuit drawing is disabled through the optional flags added to `circuit_prep.py` and `distribution.py`; their default behavior remains unchanged for `main.ipynb`.

For each qubit count, the helper writes the distributed circuit to the same bridge JSON already consumed by `schedule_pytket_dqc_with_qoala()`, then returns only compact distribution metadata to the Qoala kernel.
> **Distribution selection:** This notebook uses the updated `src/distribution.py` interface where `distribute_circuit(..., distribution_method=...)` accepts one of the 6 pytket-dqc distributor workflows listed in the configuration cell above. The selected method is forwarded into the persistent pytket kernel for every point in the sweep.


In [3]:
def prepare_distribute_and_export(n_qubits):
    n_qubits = int(n_qubits)
    remote_code = f"""import json\n\nfrom src.circuit_prep import BenchmarkLevel, get_benchmark\nimport src.circuit_prep as _circuit_prep\nfrom src.network import build_network\nimport src.distribution as _distribution\nfrom src.bridge_export import export_distributed_circuit_for_qoala\n\n# Keep results.ipynb compatible with the original project API.\n# The original functions render circuits; suppress only the render hooks.\n_circuit_prep.render_circuit_jupyter = lambda *args, **kwargs: None\n_distribution.render_circuit_jupyter = lambda *args, **kwargs: None\n\n_n_qubits = {n_qubits}\n_n_qpus = {N_QPUS}\n_qubits_per_qpu = {QUBITS_PER_QPU}\n\n# ------------------------------------------------------------\n# Compatibility fix 1: pytket OpenQASM classical width\n# ------------------------------------------------------------\n# circuit_prep parses the ORIGINAL MQT QASM before cleaning out measurements.\n# pytket's circuit_from_qasm defaults to maxwidth=32, so GHZ(33+) fails because\n# MQT writes a classical measurement register with width == n_qubits.\n# Raise only the parser limit; the existing cleaning/DQCPass pipeline is unchanged.\nfrom pytket.qasm import circuit_from_qasm as _tk_circuit_from_qasm\n\ndef _wide_circuit_from_qasm(path, *args, **kwargs):\n    kwargs.setdefault("maxwidth", max(128, _n_qubits))\n    return _tk_circuit_from_qasm(path, *args, **kwargs)\n\n_circuit_prep.circuit_from_qasm = _wide_circuit_from_qasm\n\n# ------------------------------------------------------------\n# Compatibility fix 2: missing custom analysis hooks\n# ------------------------------------------------------------\n# distribution.py already computes the official pytket-dqc ebit_cost.  The\n# original project additionally calls three custom analysis helpers that are not\n# present in this checkout.  Supply conservative fallbacks so the distribution\n# result can be returned.  None of these fallback metadata fields affects Qoala,\n# AFA-QSP, HAMFA-QSP, or the four requested plots.\nfrom pytket_dqc.utils.circuit_analysis import ebit_cost as _official_ebit_cost\n\ndef _fallback_analyse_ebits(dist_circ):\n    return dict(count=int(_official_ebit_cost(dist_circ)))\n\ndef _fallback_extract_placement(dist, n_qubits):\n    placement = getattr(dist, "placement", None)\n    if placement is None:\n        return dict()\n    if callable(placement):\n        try:\n            placement = placement()\n        except TypeError:\n            return dict()\n    return placement\n\ndef _fallback_compute_circuit_depth(dist_circ, n_ebits):\n    return int(dist_circ.depth())\n\n_distribution.analyse_ebits = _fallback_analyse_ebits\n_distribution.extract_placement = _fallback_extract_placement\n_distribution.compute_circuit_depth = _fallback_compute_circuit_depth\n\n_mqt_circuit = get_benchmark(\n    {BENCHMARK!r},\n    BenchmarkLevel.ALG,\n    _n_qubits,\n)\n\n_prepared = _circuit_prep.prepare_mqt_circuit(\n    _mqt_circuit,\n)\n_circ = _prepared.copy()\n\n_network, _links, _server_qubits = build_network(\n    n_qpus=_n_qpus,\n    qubits_per_qpu=_qubits_per_qpu,\n    print_output=False,\n)\n\n_result_dqc = _distribution.distribute_circuit(\n    _circ,\n    _network,\n    _n_qpus,\n    seed={DISTRIBUTION_SEED},\n    name={BENCHMARK!r},\n    distribution_method={DISTRIBUTION_METHOD!r},\n)\n\nif "error" in _result_dqc:\n    raise RuntimeError(\n        f"Distribution failed for {{_n_qubits}} qubits: "\n        f"{{_result_dqc.get('error_type')}}: {{_result_dqc.get('error')}}"\n    )\n\n_dist_circ = _result_dqc["distributed_circuit"]\n\nexport_distributed_circuit_for_qoala(\n    dist_circ=_dist_circ,\n    qubits_per_qpu=_qubits_per_qpu,\n    filename={str(BRIDGE_FILE)!r},\n)\n\n_metadata = {{\n    "n_qubits": int(_result_dqc["n_qubits"]),\n    "n_gates_original": int(_result_dqc["n_gates_original"]),\n    "depth_original": int(_result_dqc["depth_original"]),\n    "n_gates_distributed": int(_result_dqc["n_gates_distributed"]),\n    "depth_distributed_pytket": int(_result_dqc["depth_distributed_pytket"]),\n    "distributed_depth": int(_result_dqc["distributed_depth"]),\n    # Use the official pytket-dqc ebit_cost value for the EPR plot.\n    "epr_pairs": int(_result_dqc["ebits_builtin"]),\n    "epr_pairs_builtin": int(_result_dqc["ebits_builtin"]),\n}}\n\nwith open({PYTKET_METADATA_FILE!r}, "w") as _f:\n    json.dump(_metadata, _f, indent=2)\n"""
    PYTKET.execute(remote_code, show_output=False)
    with open(PYTKET_METADATA_FILE, 'r') as f:
        return json.load(f)

def run_one_qubit_size(n_qubits):
    distribution_meta = prepare_distribute_and_export(n_qubits)
    qoala_result = schedule_pytket_dqc_with_qoala(bridge_file=BRIDGE_FILE, single_qubit_time=SINGLE_QUBIT_TIME_NS, two_qubit_time=TWO_QUBIT_TIME_NS, starting_process_time=STARTING_PROCESS_TIME_NS, ending_process_time=ENDING_PROCESS_TIME_NS, strategy=SCHEDULING_STRATEGY, print_output=False)
    simulation_seed = PROTOCOL_SEED + int(n_qubits)
    with contextlib.redirect_stdout(io.StringIO()):
        afa_table, afa_summary = simulate_afa_qsp(qoala_result=qoala_result, p_succ=P_SUCC, time_per_trial_ns=TIME_PER_TRIAL_NS, seed=simulation_seed)
        hamfa_table, hamfa_summary = simulate_hamfa_qsp(qoala_result=qoala_result, p_succ=P_SUCC, time_per_trial_ns=TIME_PER_TRIAL_NS, t_cut_ns=T_CUT_NS, delta_t_c_ns=DELTA_T_C_NS, F_I=F_I, F_T=F_T, alpha=ALPHA, seed=simulation_seed, deadline_equal_is_blocked=DEADLINE_EQUAL_IS_BLOCKED, background_start_ns=BACKGROUND_START_NS)
    row = {**distribution_meta, 'status': 'ok', 'error': '', 'qoala_total_time_ns': float(qoala_result['total_execution_time_ns']), 'total_requests': int(afa_summary['total_requests']), 'afa_punishment_time_ns': float(afa_summary['total_punishment_time_ns']), 'hamfa_punishment_time_ns': float(hamfa_summary['total_punishment_time_ns']), 'afa_blocked_requests': int(afa_summary['total_blocked_requests']), 'hamfa_blocked_requests': int(hamfa_summary['total_blocked_requests']), 'hamfa_memory_assisted_successful': int(hamfa_summary['memory_assisted_successful']), 'hamfa_memory_assisted_not_successful': int(hamfa_summary['memory_assisted_not_successful']), 'hamfa_memory_attempted': int(hamfa_summary['memory_assisted_successful'] + hamfa_summary['memory_assisted_not_successful']), 'hamfa_all_photonic_used': int(hamfa_summary['all_photonic_used']), 'hamfa_passed_t_cut': int(hamfa_summary['passed_t_cut']), 'hamfa_direct_ap_trials': int(hamfa_summary['total_direct_AP_trials']), 'hamfa_photonic_trials_during_requests': int(hamfa_summary['total_photonic_trials_during_requests']), 'simulation_seed': simulation_seed}
    return row

def checkpoint(rows):
    pd.DataFrame(rows).sort_values('n_qubits').to_csv(CHECKPOINT_CSV, index=False)


## 4. Run the 12–100 qubit sweep

In [4]:
rows = []
for index, n_qubits in enumerate(QUBIT_COUNTS, start=1):
    print(f'[{index:02d}/{len(QUBIT_COUNTS)}] {n_qubits:3d} qubits ... ', end='', flush=True)
    try:
        row = run_one_qubit_size(n_qubits)
        rows.append(row)
        checkpoint(rows)
        print(f"done | EPR={row['epr_pairs']} | requests={row['total_requests']} | blocked AFA/HAMFA={row['afa_blocked_requests']}/{row['hamfa_blocked_requests']}")
    except Exception as exc:
        failed_row = {'n_qubits': int(n_qubits), 'status': 'failed', 'error': f'{type(exc).__name__}: {exc}'}
        rows.append(failed_row)
        checkpoint(rows)
        print(f"FAILED — {failed_row['error']}")
        if not CONTINUE_ON_ERROR:
            raise
results = pd.DataFrame(rows).sort_values('n_qubits').reset_index(drop=True)
results.to_csv(CHECKPOINT_CSV, index=False)
print(f'\nSaved {len(results)} sweep rows to {CHECKPOINT_CSV}')
display(results)


[01/22]  11 qubits ... done | EPR=1 | requests=1 | blocked AFA/HAMFA=1/0
[02/22]  16 qubits ... done | EPR=1 | requests=1 | blocked AFA/HAMFA=1/0
[03/22]  21 qubits ... done | EPR=2 | requests=2 | blocked AFA/HAMFA=2/1
[04/22]  26 qubits ... done | EPR=2 | requests=2 | blocked AFA/HAMFA=2/1
[05/22]  31 qubits ... done | EPR=4 | requests=4 | blocked AFA/HAMFA=4/2
[06/22]  36 qubits ... done | EPR=4 | requests=4 | blocked AFA/HAMFA=4/1
[07/22]  41 qubits ... done | EPR=6 | requests=6 | blocked AFA/HAMFA=6/3
[08/22]  46 qubits ... done | EPR=9 | requests=10 | blocked AFA/HAMFA=9/5
[09/22]  51 qubits ... done | EPR=7 | requests=7 | blocked AFA/HAMFA=7/2
[10/22]  56 qubits ... done | EPR=11 | requests=11 | blocked AFA/HAMFA=8/2
[11/22]  61 qubits ... done | EPR=12 | requests=11 | blocked AFA/HAMFA=11/7
[12/22]  66 qubits ... done | EPR=13 | requests=11 | blocked AFA/HAMFA=10/5
[13/22]  71 qubits ... done | EPR=14 | requests=14 | blocked AFA/HAMFA=14/7
[14/22]  76 qubits ... done | EPR=14 | 

,n_qubits,n_gates_original,depth_original,n_gates_distributed,depth_distributed_pytket,distributed_depth,epr_pairs,epr_pairs_builtin,status,error,...,afa_blocked_requests,hamfa_blocked_requests,hamfa_memory_assisted_successful,hamfa_memory_assisted_not_successful,hamfa_memory_attempted,hamfa_all_photonic_used,hamfa_passed_t_cut,hamfa_direct_ap_trials,hamfa_photonic_trials_during_requests,simulation_seed
0,11,43,23,45,24,24,1,1,ok,,...,1,0,1,0,1,0,0,0,0,53
1,16,63,33,65,34,34,1,1,ok,,...,1,0,1,0,1,0,0,0,0,58
2,21,83,43,87,45,45,2,2,ok,,...,2,1,1,1,2,1,0,28,28,63
3,26,103,53,107,55,55,2,2,ok,,...,2,1,1,1,2,1,0,34,34,68
4,31,123,63,131,67,67,4,4,ok,,...,4,2,2,2,4,2,0,29,29,73
5,36,143,73,151,77,77,4,4,ok,,...,4,1,4,0,4,0,0,0,0,78
6,41,163,83,175,89,89,6,6,ok,,...,6,3,4,2,6,2,0,14,14,83
7,46,183,93,201,103,103,9,9,ok,,...,9,5,4,4,8,6,0,81,81,88
8,51,203,103,217,110,110,7,7,ok,,...,7,2,5,2,7,2,0,49,49,93
9,56,223,113,245,124,124,11,11,ok,,...,8,2,6,4,10,5,0,94,94,98


## 5. Sanity checks and compact summary

`epr_pairs` is the project's custom `analyse_ebits()` count; `epr_pairs_builtin` is pytket-dqc's own `ebit_cost()`. Both are preserved so discrepancies are visible rather than hidden.

In [5]:
valid_results = results.loc[results['status'] == 'ok'].copy()
failed_results = results.loc[results['status'] != 'ok'].copy()
if not failed_results.empty:
    print('Failed circuit sizes:')
    display(failed_results[['n_qubits', 'error']])
else:
    print('All circuit sizes completed successfully.')
if {'epr_pairs', 'epr_pairs_builtin'}.issubset(valid_results.columns):
    epr_mismatch = valid_results.loc[valid_results['epr_pairs'] != valid_results['epr_pairs_builtin'], ['n_qubits', 'epr_pairs', 'epr_pairs_builtin']]
    if epr_mismatch.empty:
        print('Custom and pytket-dqc EPR/ebit counts agree for all completed points.')
    else:
        print('EPR/ebit count differences found (retained for inspection):')
        display(epr_mismatch)
summary_columns = ['n_qubits', 'epr_pairs', 'total_requests', 'afa_punishment_time_ns', 'hamfa_punishment_time_ns', 'hamfa_memory_assisted_successful', 'hamfa_all_photonic_used', 'afa_blocked_requests', 'hamfa_blocked_requests']
display(valid_results[summary_columns])


All circuit sizes completed successfully.
Custom and pytket-dqc EPR/ebit counts agree for all completed points.


,n_qubits,epr_pairs,total_requests,afa_punishment_time_ns,hamfa_punishment_time_ns,hamfa_memory_assisted_successful,hamfa_all_photonic_used,afa_blocked_requests,hamfa_blocked_requests
0,11,1,1,143981.0,0.000000e+00,1,0,1,0
1,16,1,1,1901810.0,0.000000e+00,1,0,1,0
2,21,2,2,6423373.0,5.505468e+06,1,1,2,1
3,26,2,2,9176107.0,6.688354e+06,1,1,2,1
4,31,4,4,10591465.0,5.306978e+06,2,2,4,2
5,36,4,4,24531625.0,1.293867e+06,4,0,4,1
6,41,6,6,23536720.0,2.862798e+06,4,2,6,3
7,46,9,10,22830761.0,1.561048e+07,4,6,9,5
8,51,7,7,35487071.0,9.618069e+06,5,2,7,2
9,56,11,11,32170396.0,1.694737e+07,6,5,8,2


## 6. plots

The plotting module uses a color-vision-deficiency-safe academic palette, serif typography, restrained grid lines, vector-friendly PDF/SVG export, and 600-dpi PNG export.

The HAMFA resource plot shows **successful memory-assisted completions** against requests for which the **all-photonic path was used**. The CSV additionally stores `hamfa_memory_attempted` if you later want to plot all memory attempts, including attempts that eventually fell back to all-photonic generation.

In [6]:
if valid_results.empty:
    raise RuntimeError('No successful sweep points are available to plot.')
figures = plot_all_results(valid_results, output_dir=FIGURE_DIR, formats=('pdf', 'svg', 'png'))
plt.show()
print('Saved figures:')
for path in sorted(FIGURE_DIR.iterdir()):
    print(' -', path.name)


Saved figures:
 - afa_average_epr_fidelity_vs_qubits.pdf
 - afa_average_epr_fidelity_vs_qubits.png
 - afa_vs_hamfa_average_epr_fidelity_vs_qubits.pdf
 - afa_vs_hamfa_average_epr_fidelity_vs_qubits.png
 - blocked_requests_vs_qubits.pdf
 - blocked_requests_vs_qubits.png
 - blocked_requests_vs_qubits.svg
 - epr_pairs_vs_qubits.pdf
 - epr_pairs_vs_qubits.png
 - epr_pairs_vs_qubits.svg
 - hamfa_average_epr_fidelity_vs_qubits.pdf
 - hamfa_average_epr_fidelity_vs_qubits.png
 - hamfa_resource_usage_vs_qubits.pdf
 - hamfa_resource_usage_vs_qubits.png
 - hamfa_resource_usage_vs_qubits.svg
 - punishment_time_vs_qubits.pdf
 - punishment_time_vs_qubits.png
 - punishment_time_vs_qubits.svg


/var/folders/40/8bqwdxj16852wcwh705klppw0000gn/T/ipykernel_56643/3041034251.py:10: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Figure definitions

1. **EPR pairs vs. circuit qubits** — `epr_pairs` on the y-axis and circuit qubit count on the x-axis.
2. **Overall punishment time vs. circuit qubits** — AFA-QSP and HAMFA-QSP together; y-axis is total punishment/delay, displayed in µs.
3. **HAMFA resource usage vs. circuit qubits** — successful memory-assisted completions and all-photonic usage together.
4. **Blocked requests vs. circuit qubits** — AFA-QSP and HAMFA-QSP together.

All raw values remain in the CSV in **ns** and integer request counts, so plotting units can be changed without rerunning the simulations.